🟢 Débutant | ⏱ 90 min | 🔑 pandas, DataFrame, filtre, groupby, nettoyage, matplotlib

# Module 10 — pandas et les graphiques

> 🎬 **Regarde d'abord la vidéo du module** (scénario `python-debutant-10`, à venir).

**Ce que tu sais déjà**
- Module 05 : une **liste** est une colonne de valeurs ; `sum`, `len`, `max`, `sorted`.
- Module 06 : une **liste de dictionnaires** est une table ; on calcule une moyenne par ville avec une boucle.
- Module 07 : ranger un calcul dans une **fonction** ; les **arguments nommés** (`nom=valeur`).
- Module 08 : lire `meteo.csv` avec `csv.DictReader`, nettoyer `meteo_brut.csv` à la main.
- Module 09 : `import`, les bibliothèques installées avec `uv pip install`.

**Ce que tu vas apprendre**
- Charger un fichier CSV en une ligne avec **pandas** et le regarder (`head`, `shape`, `dtypes`).
- Choisir des colonnes, filtrer des lignes, calculer (`mean`, `max`…), trier, regrouper par ville (`groupby`).
- Nettoyer le fichier abîmé (`str.strip`, `replace`, `to_numeric`, `dropna`, `drop_duplicates`).
- Dessiner une courbe et des barres avec **matplotlib**, et les enregistrer en image.

Pour **chaque** geste, tu verras d'abord la version « à la main » des modules précédents (boucle + liste de
dictionnaires), puis la version pandas. Tu sauras donc toujours ce que pandas fait à ta place.

**Comment lire ce notebook**
- Exécute les cellules **dans l'ordre**, une par une, avec `Shift + Entrée`.
- Les cellules 🔮 te demandent de **prédire** le résultat avant d'exécuter : écris ta réponse, puis vérifie.
- Les cellules ⚠️ plantent **exprès** : c'est pour apprendre à lire le message d'erreur.
- Pour un 🔮 dont tu ne peux pas calculer le chiffre exact (tu n'as pas toutes les données sous les yeux), prédis la **forme** du résultat : un nombre, un texte, un tableau, combien de lignes. C'est ce qui compte.
- Si tu as un doute, redémarre le kernel (Jupyter : menu *Kernel → Restart* ; VS Code : bouton *Restart* en haut du notebook) et réexécute tout depuis le début.

## 0. Préparer le dossier `sorties/`

Dans ce module, on va **écrire des fichiers** (des images de graphiques). Règle du parcours : tout ce qu'on
écrit va dans un dossier `sorties/`, à côté de ce notebook. On ne touche jamais au dossier `donnees/`.

`Path("sorties").mkdir(exist_ok=True)` crée ce dossier. C'est la même chose que `mkdir sorties` dans Git Bash
(module 00). `exist_ok=True` est un argument nommé (module 07) qui veut dire : « s'il existe déjà, ce n'est pas
une erreur, ne dis rien ».

In [ ]:
from pathlib import Path

Path("sorties").mkdir(exist_ok=True)
print(Path("sorties").exists())

## 1. Pourquoi pandas ?

Au module 08, pour calculer la température maximale moyenne de Toulouse, tu as écrit à peu près ceci.
Relis-le calmement : c'est la version « à la main ».

In [ ]:
import csv

# Version « à la main » (modules 06 et 08) : on lit le fichier ligne par ligne
releves = []
with open("../donnees/meteo.csv", encoding="utf-8") as fichier:
    for ligne in csv.DictReader(fichier):
        releves.append(ligne)

total = 0
nb_jours = 0
for releve in releves:
    if releve["ville"] == "Toulouse":
        total = total + float(releve["temp_max"])   # le CSV donne du texte : on convertit
        nb_jours = nb_jours + 1

print("Nombre de relevés :", len(releves))
print("Moyenne Toulouse :", round(total / nb_jours, 1))

Ça marche. Mais il a fallu : ouvrir le fichier, faire une boucle, convertir chaque texte en nombre, tenir un
total et un compteur. Et pour la moyenne de **chaque** ville, il faudrait un dictionnaire d'accumulateurs.

**pandas** est une **bibliothèque** (un module qu'on installe, comme `requests` au module 09) qui fait tout ça
pour toi. Imagine **Excel, mais piloté par des lignes de Python** : tu charges un tableau, tu filtres, tu fais des
moyennes, sans écrire la boucle toi-même. Elle est déjà installée dans ton `.venv` (fichier `requirements.txt`
du module 00).

On l'importe avec un **surnom** : `import pandas as pd`. Le mot `as` veut dire « sous le nom de ». Ensuite on
écrit `pd.` au lieu de `pandas.` : c'est plus court, et tout le monde fait comme ça.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
toulouse = meteo[meteo["ville"] == "Toulouse"]

print("Nombre de relevés :", len(meteo))
print("Moyenne Toulouse :", round(toulouse["temp_max"].mean(), 1))

Même résultat, en trois lignes. Ne cherche pas encore à tout comprendre : on va décortiquer chaque morceau
(`read_csv`, les crochets, `mean`) dans la suite, toujours en le comparant à la version à la main.

## 2. Le DataFrame : un tableau Excel dans Python

Un **DataFrame** (prononce « data-frème ») est le type de valeur de pandas pour un **tableau** : des lignes, des
colonnes avec un nom, comme une feuille Excel.

Tu connais déjà une table : la **liste de dictionnaires** du module 06. Chaque dictionnaire est une ligne, chaque
clé est un nom de colonne. `pd.DataFrame(liste)` transforme cette liste en DataFrame.

In [ ]:
import pandas as pd

releves = [
    {"date": "2026-09-01", "ville": "Toulouse", "temp_max": 25.9},
    {"date": "2026-09-01", "ville": "Albi", "temp_max": 26.8},
    {"date": "2026-09-02", "ville": "Toulouse", "temp_max": 23.2},
    {"date": "2026-09-02", "ville": "Albi", "temp_max": 27.0},
]

tableau = pd.DataFrame(releves)
tableau

Trois choses à remarquer :
- Les **clés** des dictionnaires sont devenues les **noms des colonnes** (en gras, en haut).
- Chaque dictionnaire est devenu une **ligne**.
- À gauche, pandas a ajouté des numéros `0, 1, 2, 3` : c'est l'**index**, les étiquettes des lignes. Comme les
  numéros de ligne d'Excel, sauf qu'ici on commence à 0 (comme les positions d'une liste, module 05).

Autre détail : on n'a pas écrit `print(tableau)`. Dans Jupyter, **la dernière ligne d'une cellule s'affiche toute
seule**. Pour un DataFrame, cet affichage automatique est plus joli que `print` : un vrai tableau.

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

releves = [
    {"ville": "Nîmes", "pluie_mm": 0.0},
    {"ville": "Tarbes", "pluie_mm": 8.1},
    {"ville": "Albi", "pluie_mm": 5.0},
]
petit = pd.DataFrame(releves)
print(len(petit))

<details><summary>👀 Voir l'explication</summary>

`3`. `len` d'un DataFrame donne le **nombre de lignes**, comme `len` d'une liste donnait le nombre d'éléments.
Ici, 3 dictionnaires, donc 3 lignes.

</details>

## 3. Charger un CSV : `pd.read_csv`

`pd.read_csv(chemin)` lit un fichier CSV et renvoie un DataFrame. Il fait en une ligne tout le travail du
module 08 : ouvrir le fichier, lire l'en-tête, découper chaque ligne, **et convertir les nombres** (fini les
`float(...)` partout).

Le chemin est le même qu'avant : `../donnees/meteo.csv` (`..` = remonter d'un dossier, module 08).

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
meteo.head()

`meteo.head()` affiche les **5 premières lignes**. C'est le premier réflexe devant un nouveau fichier : jeter un
œil au début. `head(3)` en affiche 3, `head(10)` en affiche 10.

Rappel du module 02 : un **point** suivi d'un nom et de parenthèses, c'est une **méthode**, une fonction
« attachée » à une valeur. Tu as déjà écrit `"texte".upper()`. Ici, `meteo.head()` : la méthode `head` du
DataFrame `meteo`.

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
print(len(meteo.head(2)))

<details><summary>👀 Voir l'explication</summary>

`2`. `head(2)` renvoie un **nouveau DataFrame** qui ne contient que les 2 premières lignes ; `len` compte ses lignes.

</details>

### Combien de lignes, combien de colonnes : `shape`

`meteo.shape` donne la **taille** du tableau sous forme de **tuple** (module 06) : `(nombre de lignes, nombre de
colonnes)`. Attention : **pas de parenthèses** après `shape`. Ce n'est pas une méthode, c'est une information
rangée dans le DataFrame (on dit un **attribut**). Tu en reparleras au module suivant.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
print(meteo.shape)
nb_lignes, nb_colonnes = meteo.shape     # déballage d'un tuple (module 06)
print(nb_lignes, "lignes et", nb_colonnes, "colonnes")

### Le type de chaque colonne : `dtypes`

`meteo.dtypes` (encore sans parenthèses) donne le **type de chaque colonne** (« dtype » = *data type*, type de
donnée). Comme `type()` du module 01, mais pour une colonne entière.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
print(meteo.dtypes)

Comment lire ce résultat :
- `float64` : nombre à virgule (le `float` du module 01). Le `64` est la taille en mémoire : ignore-le.
- `int64` : nombre entier (`int`).
- `str` : du texte. Sur une version plus ancienne de pandas, le texte s'affiche `object` : c'est pareil pour nous.
- La dernière ligne `dtype: object` parle de la liste des types elle-même : ignore-la.

pandas a deviné tout seul que `temp_max` est un nombre. Au module 08, `csv.DictReader` te donnait du texte
(`"25.9"`) et tu devais convertir.

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
print(meteo.shape[1])

<details><summary>👀 Voir l'explication</summary>

`6`. `shape` est le tuple `(180, 6)` ; `[1]` prend l'élément en position 1, donc le nombre de colonnes :
`date, ville, temp_min, temp_max, pluie_mm, vent_kmh`.

</details>

### Un autre séparateur : `sep=";"`

`meteo_brut.csv` sépare ses colonnes par des **points-virgules**. On le dit à pandas avec l'argument nommé
`sep` (pour *separator*, séparateur). Sans lui, pandas chercherait des virgules et verrait une seule colonne.

In [ ]:
import pandas as pd

brut = pd.read_csv("../donnees/meteo_brut.csv", sep=";")
print(brut.shape)
brut.head(8)

Tu reconnais les défauts du module 08 : `23,9` avec une virgule, `  montpellier ` avec des espaces, et une case
`NaN` dans `pluie_mm`. On les nettoiera à la section 9.

⚠️ **Erreur fréquente : un chemin faux.** Exécute la cellule suivante : elle plante exprès.

In [ ]:
import pandas as pd

meteo = pd.read_csv("donnees/meteo.csv")

**Lisons le message.** Les erreurs de pandas sont **longues** : beaucoup de lignes parlent de l'intérieur de
pandas. Ne te décourage pas, la règle du module 08 reste vraie : **va directement à la dernière ligne.**

- Dernière ligne : `FileNotFoundError: [Errno 2] No such file or directory: 'donnees/meteo.csv'`.
- Le **type d'erreur** est `FileNotFoundError` : fichier introuvable (tu l'as rencontré au module 08).
- Le message cite le chemin qui a été essayé : `'donnees/meteo.csv'`.
- Remonte tout en haut du message : la flèche `---->` montre **ta** ligne, celle qui a déclenché l'erreur.

**Cause** : le notebook est dans le dossier du module ; `donnees/` est un cran au-dessus. **Correction** : ajouter
`../` devant, soit `pd.read_csv("../donnees/meteo.csv")`.

## 4. Choisir une colonne

**À la main** (module 06), pour avoir toutes les températures maximales, tu construisais une liste avec une boucle.

In [ ]:
releves = [
    {"ville": "Toulouse", "temp_max": 25.9},
    {"ville": "Albi", "temp_max": 26.8},
    {"ville": "Tarbes", "temp_max": 22.0},
]
temperatures = []
for releve in releves:
    temperatures.append(releve["temp_max"])
print(temperatures)

**Avec pandas**, on écrit le nom de la colonne entre crochets, comme une clé de dictionnaire :
`meteo["temp_max"]`. Le résultat est une **Series** (prononce « siri-z ») : **une seule colonne**, avec son index à
gauche. Une Series ressemble à une liste… qui garderait les étiquettes des lignes.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
temperatures = meteo["temp_max"]
print(temperatures.head())

Lecture de l'affichage : à gauche l'index (`0` à `4`), à droite les valeurs. La dernière ligne rappelle le nom
de la colonne (`Name: temp_max`) et son type (`dtype: float64`).

Pour **plusieurs colonnes**, on met une **liste** de noms entre les crochets. D'où les **doubles crochets** :
les crochets extérieurs veulent dire « choisis », les crochets intérieurs sont la liste (module 05).
Le résultat est un DataFrame plus étroit.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
extrait = meteo[["date", "ville", "temp_max"]]
extrait.head(3)

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
extrait = meteo[["ville", "pluie_mm"]]
print(extrait.shape)

<details><summary>👀 Voir l'explication</summary>

`(180, 2)`. On garde **toutes les lignes** (180) mais seulement **2 colonnes**.

</details>

### Récupérer une valeur précise : `list(...)[0]`

Parfois on veut **une** valeur, par exemple la première de la colonne. Astuce avec ce que tu connais déjà :
`list(colonne)` transforme la Series en liste Python (module 05), puis `[0]` prend la première case.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
premiere_ville = list(meteo["ville"])[0]
print(premiere_ville)

⚠️ **Erreur fréquente : un nom de colonne mal écrit.**

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
meteo["temp max"]

**Lisons le message.** Encore un long message : file à la **dernière ligne**.

- Dernière ligne : `KeyError: 'temp max'`.
- `KeyError`, tu le connais (module 06) : « cette clé n'existe pas ». Ici la clé, c'est le nom de colonne.
- Le message cite exactement ce que tu as tapé : `'temp max'`, avec un **espace**.

**Correction** : le vrai nom est `temp_max`, avec un tiret bas. En cas de doute, affiche la liste des colonnes :
`meteo.columns` (sans parenthèses, un attribut comme `shape`) contient les noms des colonnes, et `list(...)` en
fait une liste lisible. Python compare les noms **au caractère près** (majuscules et accents compris).

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
print(list(meteo.columns))

## 5. Calculer sur une colonne : `mean`, `max`, `min`, `sum`, `count`

**À la main** (module 05) : `sum(liste) / len(liste)` pour une moyenne, `max(liste)` pour le maximum.

**Avec pandas**, la colonne a ses propres méthodes :

| Méthode | Ce qu'elle calcule | Version à la main |
|---|---|---|
| `.mean()` | la moyenne | `sum(liste) / len(liste)` |
| `.max()` / `.min()` | la plus grande / la plus petite valeur | `max(liste)` / `min(liste)` |
| `.sum()` | le total | `sum(liste)` |
| `.count()` | le nombre de cases **remplies** | `len(liste)` (sans les cases vides) |

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
print("Moyenne :", meteo["temp_max"].mean())
print("Maximum :", meteo["temp_max"].max())
print("Minimum :", meteo["temp_min"].min())
print("Pluie totale :", meteo["pluie_mm"].sum())
print("Cases remplies :", meteo["pluie_mm"].count())

La moyenne a beaucoup de décimales. `round(valeur, 1)` (module 01) arrondit à 1 chiffre après la virgule, comme
d'habitude.

🔮 **Prédis** : Avant d'exécuter : la cellule va-t-elle afficher un tableau, une colonne ou un seul nombre ? Avec combien de chiffres après la virgule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
moyenne_vent = round(meteo["vent_kmh"].mean(), 1)
print(moyenne_vent)

<details><summary>👀 Voir l'explication</summary>

**Un seul nombre, avec un chiffre après la virgule** (ici `31.1`). `meteo["vent_kmh"]` choisit la
colonne, `.mean()` fait la moyenne des 180 valeurs, `round(..., 1)` garde un chiffre après la virgule.

</details>

### Calculer une nouvelle colonne

**À la main**, pour l'écart de température de chaque jour, il fallait une boucle et un `append`.

**Avec pandas**, on écrit le calcul **une seule fois, sur les colonnes entières** : pandas l'applique ligne par
ligne tout seul. Et on range le résultat dans une **nouvelle colonne**, avec la même écriture qu'on utilisait pour
ajouter une clé à un dictionnaire (module 06) : `meteo["ecart"] = ...`.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
meteo["ecart"] = meteo["temp_max"] - meteo["temp_min"]
meteo["temp_max_f"] = meteo["temp_max"] * 9 / 5 + 32      # conversion °C -> °F (module 01)
meteo[["ville", "temp_min", "temp_max", "ecart", "temp_max_f"]].head(3)

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

releves = [
    {"ville": "Albi", "temp_min": 12.0, "temp_max": 26.0},
    {"ville": "Tarbes", "temp_min": 10.0, "temp_max": 22.0},
]
petit = pd.DataFrame(releves)
petit["ecart"] = petit["temp_max"] - petit["temp_min"]
print(petit["ecart"].max())

<details><summary>👀 Voir l'explication</summary>

`14.0`. L'écart vaut 14.0 pour Albi (26 − 12) et 12.0 pour Tarbes (22 − 10) ; le maximum est 14.0.

</details>

## 6. Filtrer des lignes : le filtre booléen

**À la main** (modules 03 et 06) : une boucle, un `if`, et on garde les lignes qui conviennent.

In [ ]:
releves = [
    {"ville": "Toulouse", "temp_max": 25.9},
    {"ville": "Albi", "temp_max": 26.8},
    {"ville": "Toulouse", "temp_max": 23.2},
]
seulement_toulouse = []
for releve in releves:
    if releve["ville"] == "Toulouse":
        seulement_toulouse.append(releve)
print(seulement_toulouse)

**Avec pandas**, cela se fait en deux temps.

**Temps 1 : poser la question à toute la colonne.** `meteo["ville"] == "Toulouse"` compare **chaque** case à
`"Toulouse"`. Le résultat est une colonne de `True` / `False` (des **booléens**, module 03), une réponse par ligne.
On appelle cette colonne un **masque** : comme un pochoir posé sur le tableau, il ne laisse passer que les lignes
`True`.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
masque = meteo["ville"] == "Toulouse"
print(masque.head(7))

**Temps 2 : appliquer le masque.** On met le masque entre crochets : `meteo[masque]`. pandas garde seulement les
lignes où le masque vaut `True`. On écrit souvent les deux temps sur une seule ligne :
`meteo[meteo["ville"] == "Toulouse"]`. Le nom `meteo` apparaît deux fois : c'est normal.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
toulouse = meteo[meteo["ville"] == "Toulouse"]
print(len(toulouse), "jours à Toulouse")
toulouse.head(3)

Toutes les comparaisons du module 03 marchent : `==`, `!=`, `>`, `>=`, `<`, `<=`. Par exemple, les jours à plus
de 28 °C, toutes villes confondues :

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
tres_chaud = meteo[meteo["temp_max"] > 28]
print(len(tres_chaud), "relevés au-dessus de 28 °C")

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
tarbes = meteo[meteo["ville"] == "Tarbes"]
print(len(tarbes))

<details><summary>👀 Voir l'explication</summary>

`30`. Le fichier contient 30 jours de septembre pour chacune des 6 villes : 6 × 30 = 180 lignes, dont 30 pour Tarbes.

</details>

### Comparer des dates écrites en texte

`<`, `<=`, `>`, `>=` marchent aussi sur du **texte** : Python compare lettre par lettre, dans l'ordre du
dictionnaire (`"Albi" < "Nîmes"` vaut `True`). Pour des dates écrites `AAAA-MM-JJ` (année, puis mois, puis jour),
l'ordre du dictionnaire est **exactement** l'ordre du calendrier : `"2026-09-05" <= "2026-09-07"` vaut `True`.
On peut donc filtrer une période sur la colonne `date`, comme sur des nombres.

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
debut_albi = meteo[meteo["ville"] == "Albi"]
debut_albi = debut_albi[debut_albi["date"] <= "2026-09-05"]
print(len(debut_albi), list(debut_albi["date"])[-1])

<details><summary>👀 Voir l'explication</summary>

`5 2026-09-05`. On garde d'abord Albi (30 lignes), puis les dates jusqu'au 5 septembre inclus (`<=`) : du 1er au 5,
soit 5 lignes. La dernière date gardée est bien `2026-09-05`.

</details>

### Deux conditions : `&` (et), `|` (ou)

Pour combiner deux conditions, pandas n'utilise **pas** `and` et `or` mais des symboles :
- `&` veut dire **et** (les deux conditions vraies) ;
- `|` veut dire **ou** (au moins une vraie). Sur un clavier français : `Alt Gr + 6`.

Et chaque condition se met **entre parenthèses**. Sans elles, Python calcule dans le mauvais ordre (la priorité
des opérations, module 01).

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
toulouse_pluie = meteo[(meteo["ville"] == "Toulouse") & (meteo["pluie_mm"] > 0)]
print(len(toulouse_pluie), "jours de pluie à Toulouse")

cote = meteo[(meteo["ville"] == "Perpignan") | (meteo["ville"] == "Montpellier")]
print(len(cote), "relevés sur la côte")

🔮 **Prédis** : Avant d'exécuter : la cellule va-t-elle afficher un tableau ou un nombre ? Ce nombre peut-il dépasser 30 ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
nimes_chaud = meteo[(meteo["ville"] == "Nîmes") & (meteo["temp_max"] >= 29)]
print(len(nimes_chaud))

<details><summary>👀 Voir l'explication</summary>

**Un nombre, au plus 30** : Nîmes n'a que 30 jours dans le fichier, et le filtre ne peut qu'en enlever (ici
`7`). On garde les lignes où **les deux** conditions sont vraies :
la ville est Nîmes **et** la température maximale atteint au moins 29 °C.

</details>

⚠️ **Erreur fréquente : écrire `and` au lieu de `&`.**

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
meteo[(meteo["ville"] == "Toulouse") and (meteo["temp_max"] > 25)]

**Lisons le message.** Dernière ligne :
`ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().`

- Le type est `ValueError` : une valeur qui ne convient pas (module 02).
- Traduction : « la valeur de vérité d'une Series est ambiguë ». `and` veut **un seul** `True` ou `False`. Or
  `meteo["ville"] == "Toulouse"` est une colonne de 180 réponses. Python ne sait pas laquelle prendre.
- La fin du message propose des solutions compliquées (`a.any()`…) : **ignore-les**, ce n'est pas notre cas.

**Correction** : remplacer `and` par `&` (et `or` par `|`), en gardant les parenthèses autour de chaque condition.

## 7. Trier : `sort_values`

**À la main** (module 05), `sorted(liste)` trie une liste de nombres. Mais trier une liste de **dictionnaires** par
une de leurs clés, tu ne l'as pas fait : c'est compliqué.

**Avec pandas**, `meteo.sort_values("temp_max")` renvoie le tableau trié selon la colonne `temp_max`, du plus
petit au plus grand. Pour l'ordre inverse (du plus grand au plus petit), on ajoute l'argument nommé
`ascending=False` (« croissant : non »).

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
classement = meteo.sort_values("temp_max", ascending=False)
classement.head(3)

Les trois journées les plus chaudes du mois. Remarque l'index : les numéros ne sont plus dans l'ordre. Chaque
ligne a **gardé son étiquette** d'origine ; c'est normal.

Pour récupérer la ville de la journée la plus chaude, on combine avec l'astuce de la section 4 :

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
classement = meteo.sort_values("temp_max", ascending=False)
ville_record = list(classement["ville"])[0]
date_record = list(classement["date"])[0]
print("Record :", ville_record, "le", date_record)

⚠️ **Piège : un tri qui n'est pas rangé ne change rien.** `sort_values` ne modifie pas le tableau : il en
**renvoie une copie triée**. Si tu ne ranges pas cette copie dans une variable, elle est perdue.

🔮 **Prédis** : Avant d'exécuter : les deux listes affichées seront-elles dans le même ordre ? Laquelle sera triée de la plus chaude à la moins chaude ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
petit = meteo.head(3)
petit.sort_values("temp_max", ascending=False)     # résultat pas rangé
print(list(petit["temp_max"]))

petit = petit.sort_values("temp_max", ascending=False)   # résultat rangé dans petit
print(list(petit["temp_max"]))

<details><summary>👀 Voir l'explication</summary>

**Non, seule la deuxième est triée.** D'abord `[25.9, 29.2, 29.2]` : l'ordre d'origine, rien n'a changé. Puis
`[29.2, 29.2, 25.9]` : cette fois, le tableau trié a été **rangé** dans `petit`. Retiens la forme :
`tableau = tableau.sort_values(...)`.

</details>

🔮 **Prédis** : Avant d'exécuter : la valeur affichée sera-t-elle la plus petite ou la plus grande `temp_min` du fichier ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
classement = meteo.sort_values("temp_min")
print(list(classement["temp_min"])[0])

<details><summary>👀 Voir l'explication</summary>

**La plus petite** (ici `4.9`). Sans `ascending=False`, le tri est **croissant** : la première ligne
a la plus petite `temp_min`. C'est donc la même valeur que `meteo["temp_min"].min()`.

</details>

## 8. Regrouper : `groupby`

C'est le geste le plus puissant de ce module. **À la main** (module 06), pour la moyenne **de chaque ville**, il
fallait deux dictionnaires d'accumulateurs (un total et un compteur par ville). Relis-le :

In [ ]:
import csv

totaux = {}
compteurs = {}
with open("../donnees/meteo.csv", encoding="utf-8") as fichier:
    for ligne in csv.DictReader(fichier):
        ville = ligne["ville"]
        totaux[ville] = totaux.get(ville, 0) + float(ligne["temp_max"])
        compteurs[ville] = compteurs.get(ville, 0) + 1

for ville in totaux:
    print(ville, round(totaux[ville] / compteurs[ville], 1))

**Avec pandas**, la même chose se dit presque en français :
« **regroupe** les lignes **par** ville, prends la colonne `temp_max`, fais la **moyenne** de chaque groupe ».

```python
meteo.groupby("ville")["temp_max"].mean()
```

`groupby` (« grouper par ») fait des **paquets** de lignes : un paquet par ville, comme si tu triais des fiches en
six piles. Ensuite, le calcul (`mean`, `max`, `sum`, `count`…) est fait **pile par pile**.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
moyennes = meteo.groupby("ville")["temp_max"].mean().round(1)
print(moyennes)

Le résultat est une **Series** dont l'**index** est le nom des villes (rangées par ordre alphabétique). Note
`.round(1)` : la version pandas de `round`, appliquée à toute la colonne d'un coup.

Une Series dont l'index est un nom se manipule **comme un dictionnaire** (module 06) :
- `moyennes["Toulouse"]` lit une valeur, comme une clé ;
- `for ville, valeur in moyennes.items():` parcourt les couples (étiquette, valeur), comme `.items()` d'un dict.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
moyennes = meteo.groupby("ville")["temp_max"].mean().round(1)
print("Toulouse :", moyennes["Toulouse"])

for ville, valeur in moyennes.items():
    print(f"{ville} : {valeur} °C")

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
cumul = meteo.groupby("ville")["pluie_mm"].sum().round(1)
print(len(cumul))

<details><summary>👀 Voir l'explication</summary>

`6`. Un groupe par ville, donc 6 valeurs. `groupby` ne garde qu'**une ligne par groupe**.

</details>

### Trier le résultat, plusieurs colonnes à la fois

- Une Series se trie aussi avec `sort_values` (sans nom de colonne, puisqu'il n'y en a qu'une).
- `list(serie.index)[0]` donne l'étiquette de la première ligne : par exemple la ville en tête du classement.
- Avec des doubles crochets, on calcule plusieurs colonnes d'un coup ; le résultat est un DataFrame.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
moyennes = meteo.groupby("ville")["temp_max"].mean().round(1)
classement = moyennes.sort_values(ascending=False)
print(classement)
print("Ville la plus chaude :", list(classement.index)[0])

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
tableau = meteo.groupby("ville")[["temp_min", "temp_max"]].mean().round(1)
tableau

On peut ensuite **ajouter une colonne** à ce tableau, avec la même écriture qu'à la section 5. pandas range
chaque valeur sur la bonne ville grâce à l'index. Pour lire **une case** du tableau, on choisit d'abord la colonne,
puis la ville : `tableau["pluie_totale"]["Albi"]` (colonne, puis étiquette de ligne).

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
tableau = meteo.groupby("ville")[["temp_min", "temp_max"]].mean().round(1)
tableau["pluie_totale"] = meteo.groupby("ville")["pluie_mm"].sum().round(1)
print("Pluie totale d'Albi :", tableau["pluie_totale"]["Albi"])
tableau.sort_values("temp_max", ascending=False)

🔮 **Prédis** : Avant d'exécuter : `records` contient-il une valeur par ligne du fichier, ou une par ville ? Et `print` va-t-il afficher un tableau ou un seul nombre ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
records = meteo.groupby("ville")["vent_kmh"].max()
print(records["Tarbes"])

<details><summary>👀 Voir l'explication</summary>

**Une valeur par ville** (6 en tout), et `print` affiche **un seul nombre** : le vent le plus fort de Tarbes (ici
`52`). `groupby("ville")["vent_kmh"].max()` donne le vent le plus fort de
chaque ville ; `["Tarbes"]` lit la valeur de Tarbes, comme une clé de dictionnaire.

</details>

## 9. Nettoyer le fichier abîmé

Au module 08, tu as nettoyé `meteo_brut.csv` ligne par ligne, avec des `.strip()`, des `.replace()` et des `if`.
Voici les mêmes gestes, mais **sur une colonne entière d'un coup**. On repart du fichier brut à chaque cellule,
pour que chaque exemple soit indépendant.

### 9.1 Les espaces et les minuscules dans `ville`

**À la main** : `"  montpellier ".strip()` donnait `"montpellier"` (module 02).

**Avec pandas**, les méthodes de texte d'une colonne sont rangées derrière `.str` : `brut["ville"].str.strip()`
applique `.strip()` à chaque case. Puis on **remplace** la colonne par sa version propre.

In [ ]:
import pandas as pd

brut = pd.read_csv("../donnees/meteo_brut.csv", sep=";")
print("Avant :", sorted(set(brut["ville"])))

brut["ville"] = brut["ville"].str.strip()
print("Après :", sorted(set(brut["ville"])))

Les espaces ont disparu (`set` du module 06 dédoublonne, `sorted` trie). Il reste des noms en minuscules :
`montpellier` au lieu de `Montpellier`.

`.replace(correspondances)` remplace des **valeurs entières** à l'aide d'un **dictionnaire** : chaque clé est
remplacée par sa valeur. C'est un dictionnaire de traduction, comme un glossaire.

In [ ]:
import pandas as pd

brut = pd.read_csv("../donnees/meteo_brut.csv", sep=";")
corrections = {
    "toulouse": "Toulouse", "montpellier": "Montpellier", "nîmes": "Nîmes",
    "perpignan": "Perpignan", "albi": "Albi", "tarbes": "Tarbes",
}
brut["ville"] = brut["ville"].str.strip().replace(corrections)
print(sorted(set(brut["ville"])))

Six villes, bien écrites. Note qu'on a **enchaîné** deux méthodes : d'abord `.str.strip()`, puis `.replace(...)`
sur le résultat. On lit de gauche à droite.

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

releves = [{"ville": " albi "}, {"ville": "Albi"}, {"ville": "albi"}]
petit = pd.DataFrame(releves)
petit["ville"] = petit["ville"].str.strip().replace({"albi": "Albi"})
print(len(set(petit["ville"])))

<details><summary>👀 Voir l'explication</summary>

`1`. Après `.str.strip()`, il reste `"albi"`, `"Albi"`, `"albi"`. Le `replace` change les `"albi"` en `"Albi"`.
Le `set` ne contient plus qu'une valeur : `{"Albi"}`.

</details>

### 9.2 La virgule décimale dans `temp_max`

Regarde `brut.dtypes` : `temp_max` est du **texte** (`str`), à cause de valeurs comme `23,9`. Python veut un point.
(Pour une seule colonne, on écrit `.dtype`, au singulier : `brut["temp_max"].dtype`.)

**À la main** : `float("23,9".replace(",", "."))` (module 02).

**Avec pandas**, deux étapes :
1. `.str.replace(",", ".")` : remplace la virgule par un point **à l'intérieur** du texte de chaque case.
   (À ne pas confondre avec `.replace(dictionnaire)` de la section 9.1, qui remplace la **case entière**.)
2. `pd.to_numeric(colonne)` : convertit une colonne de texte en nombres. C'est le `float()` du module 02, pour toute
   la colonne.

In [ ]:
import pandas as pd

brut = pd.read_csv("../donnees/meteo_brut.csv", sep=";")
print("Avant :", brut["temp_max"].dtype)

brut["temp_max"] = pd.to_numeric(brut["temp_max"].str.replace(",", "."))
print("Après :", brut["temp_max"].dtype)
print("Moyenne :", round(brut["temp_max"].mean(), 1))

⚠️ **Erreur fréquente : convertir sans avoir remplacé la virgule.**

In [ ]:
import pandas as pd

brut = pd.read_csv("../donnees/meteo_brut.csv", sep=";")
brut["temp_max"] = pd.to_numeric(brut["temp_max"])

**Lisons le message.** Dernière ligne : `ValueError: Unable to parse string "23,9" at position 3`.

- `ValueError` : comme `float("23,9")` au module 02, la **valeur** n'a pas la bonne forme.
- Traduction : « impossible de lire le texte "23,9" à la position 3 ».
- La position, c'est l'**index** de la ligne fautive : la ligne `3` (Perpignan, le 1er septembre). Très pratique
  pour aller voir avec `brut.head(5)`.

**Correction** : faire d'abord `.str.replace(",", ".")`, puis `pd.to_numeric(...)`, comme dans la cellule d'avant.

### 9.3 Les cases vides : `NaN`, `isna`, `dropna`

Dans `pluie_mm`, certaines cases sont vides ; dans `vent_kmh`, certaines contiennent `N/A` (*not available*, « non
disponible »). pandas reconnaît les deux et les remplace par **`NaN`** (*Not a Number*, « pas un nombre ») : la
façon pandas d'écrire **« case vide »**. Au module 08, tu devais tester ces cas avec des `if`.

- `colonne.isna()` donne un masque : `True` là où la case est vide. On peut s'en servir comme filtre (section 6).
- `.mean()`, `.sum()`… **ignorent** les cases vides. C'est pour ça que `.count()` peut être plus petit que `len`.
- `tableau.dropna()` (*drop* = laisser tomber, *na* = non disponible) renvoie le tableau **sans les lignes qui ont au
  moins une case vide**.

In [ ]:
import pandas as pd

brut = pd.read_csv("../donnees/meteo_brut.csv", sep=";")
pluie_vide = brut[brut["pluie_mm"].isna()]
print(len(pluie_vide), "lignes sans pluie mesurée")
print("len :", len(brut["pluie_mm"]), "| count :", brut["pluie_mm"].count())

sans_trou = brut.dropna()
print(len(brut), "lignes avant dropna,", len(sans_trou), "après")

🔮 **Prédis** : Avant d'exécuter : la cellule va-t-elle afficher un tableau ou un nombre ? Ce nombre sera-t-il plus petit, égal ou plus grand que `len(brut)` ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd

brut = pd.read_csv("../donnees/meteo_brut.csv", sep=";")
vent_vide = brut[brut["vent_kmh"].isna()]
print(len(vent_vide))

<details><summary>👀 Voir l'explication</summary>

**Un nombre, plus petit que `len(brut)`** : le filtre ne garde qu'une partie des lignes (ici
`6`). Le masque `brut["vent_kmh"].isna()` vaut `True` sur les lignes où le vent est
`N/A` ; le filtre ne garde que ces lignes, et `len` les compte.

</details>

### 9.4 Les doublons : `drop_duplicates`

Certaines lignes apparaissent deux fois. **À la main**, tu aurais gardé un `set` des lignes déjà vues (module 06).

**Avec pandas** : `tableau.drop_duplicates()` renvoie le tableau sans les lignes **strictement identiques** à une
ligne déjà vue. Attention : `"Albi"` et `" albi "` ne sont pas identiques. Il faut donc **nettoyer les villes
d'abord**, puis enlever les doublons.

In [ ]:
import pandas as pd

brut = pd.read_csv("../donnees/meteo_brut.csv", sep=";")
corrections = {
    "toulouse": "Toulouse", "montpellier": "Montpellier", "nîmes": "Nîmes",
    "perpignan": "Perpignan", "albi": "Albi", "tarbes": "Tarbes",
}
brut["ville"] = brut["ville"].str.strip().replace(corrections)
brut["temp_max"] = pd.to_numeric(brut["temp_max"].str.replace(",", "."))
propre = brut.drop_duplicates()

print(len(brut), "lignes avant,", len(propre), "après drop_duplicates")
print("Moyenne temp_max :", round(propre["temp_max"].mean(), 2))

180 lignes : on retrouve exactement le nombre de lignes de `meteo.csv`. Le fichier brut est réparé (il reste des
cases vides, qu'on peut enlever avec `dropna()` si on en a besoin).

### 9.5 Revenir à une liste de dictionnaires : `to_dict("records")`

Le chemin inverse de la section 2 existe : `tableau.to_dict("records")` transforme un DataFrame en **liste de
dictionnaires** (*records* = « fiches »). Pratique pour réutiliser un code des modules précédents.

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
tarbes = meteo[meteo["ville"] == "Tarbes"].head(2)
fiches = tarbes.to_dict("records")
print(fiches)
print(fiches[0]["temp_max"])

## 10. Dessiner des graphiques avec matplotlib

**matplotlib** est la bibliothèque de graphiques de Python (elle aussi installée par `requirements.txt`). On
importe sa partie « tracé », `pyplot`, sous le surnom `plt` : `import matplotlib.pyplot as plt`.

Une **figure** est la feuille sur laquelle on dessine. Les **axes** sont les deux règles graduées : l'axe
horizontal (abscisses, *x*) et l'axe vertical (ordonnées, *y*). Le principe : tu donnes à matplotlib la liste des
*x* et la liste des *y*, il place les points.

### 10.1 Une courbe : `plt.plot`

Pour la température maximale de Toulouse jour après jour : en *x* les numéros de jour 1 à 30 (`range`, module 04),
en *y* la colonne `temp_max`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

meteo = pd.read_csv("../donnees/meteo.csv")
toulouse = meteo[meteo["ville"] == "Toulouse"]
jours = list(range(1, len(toulouse) + 1))      # 1, 2, ..., 30

plt.plot(jours, toulouse["temp_max"])
plt.show()

Une courbe brute. Il manque l'essentiel : **de quoi parle-t-on ?** Un graphique sans titre ni légende des axes ne
veut rien dire pour Inès. Trois fonctions :
- `plt.title("...")` : le titre en haut ;
- `plt.xlabel("...")` : la légende de l'axe horizontal (*x label*) ;
- `plt.ylabel("...")` : la légende de l'axe vertical.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

meteo = pd.read_csv("../donnees/meteo.csv")
toulouse = meteo[meteo["ville"] == "Toulouse"]
jours = list(range(1, len(toulouse) + 1))

plt.plot(jours, toulouse["temp_max"])
plt.title("Toulouse : température maximale en septembre 2026")
plt.xlabel("Jour du mois")
plt.ylabel("Température (°C)")
plt.show()

### 10.2 Enregistrer l'image : `plt.savefig`

`plt.savefig("sorties/nom.png")` enregistre le graphique dans un fichier image (`.png`). **Ordre important :**
`savefig` **avant** `show`. Dans un fichier `.py`, `plt.show()` affiche puis **efface** la figure : un `savefig`
placé après enregistrerait une image blanche. Dans Jupyter, le graphique s'affiche même sans `plt.show()`, mais
on l'écrit quand même : c'est indispensable dans un fichier `.py`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

Path("sorties").mkdir(exist_ok=True)
meteo = pd.read_csv("../donnees/meteo.csv")
toulouse = meteo[meteo["ville"] == "Toulouse"]
jours = list(range(1, len(toulouse) + 1))

plt.plot(jours, toulouse["temp_max"])
plt.title("Toulouse : température maximale en septembre 2026")
plt.xlabel("Jour du mois")
plt.ylabel("Température (°C)")
plt.savefig("sorties/cours_toulouse.png")
plt.show()

print(Path("sorties/cours_toulouse.png").exists())

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

meteo = pd.read_csv("../donnees/meteo.csv")
albi = meteo[meteo["ville"] == "Albi"]
jours = list(range(1, len(albi) + 1))
print(len(jours), jours[0], jours[-1])
plt.plot(jours, albi["temp_min"])
plt.show()

<details><summary>👀 Voir l'explication</summary>

D'abord `30 1 30` : 30 jours, du jour 1 au jour 30 (`range(1, 31)` s'arrête **avant** 31, module 04). Puis la
courbe des températures minimales d'Albi. Les deux listes (*x* et *y*) ont la même longueur, 30 : c'est obligatoire.

</details>

### 10.3 Des barres : `plt.bar`

Pour **comparer des villes**, une courbe n'a pas de sens (il n'y a pas d'ordre entre les villes). On utilise des
**barres** : `plt.bar(etiquettes, hauteurs)`. Les étiquettes sont les noms des villes (l'index de la Series
`groupby`), les hauteurs sont les valeurs.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

Path("sorties").mkdir(exist_ok=True)
meteo = pd.read_csv("../donnees/meteo.csv")
moyennes = meteo.groupby("ville")["temp_max"].mean().round(1).sort_values(ascending=False)

plt.bar(list(moyennes.index), list(moyennes))
plt.title("Température maximale moyenne, septembre 2026")
plt.xlabel("Ville")
plt.ylabel("Température (°C)")
plt.savefig("sorties/cours_barres.png")
plt.show()

`list(moyennes.index)` donne la liste des villes, `list(moyennes)` la liste des valeurs : deux listes de même
longueur, comme pour `plot`. Les barres sont dans l'ordre du `sort_values` : de la plus chaude à la moins chaude.

🔮 **Prédis** : Avant d'exécuter : combien de barres vas-tu voir, et dans quel ordre seront les villes ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

meteo = pd.read_csv("../donnees/meteo.csv")
cumul = meteo.groupby("ville")["pluie_mm"].sum().round(1)
plt.bar(list(cumul.index), list(cumul))
plt.title("Pluie totale")
plt.show()

<details><summary>👀 Voir l'explication</summary>

**Six barres**, une par ville, rangées par **ordre alphabétique** : `groupby` range les villes ainsi, et on n'a pas
trié. Ici, la plus haute est Perpignan avec
136.3 mm.

</details>

⚠️ **Erreur fréquente : enregistrer dans un dossier qui n'existe pas.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

meteo = pd.read_csv("../donnees/meteo.csv")
plt.bar(["Albi", "Tarbes"], [22.7, 20.6])
plt.savefig("resultats/barres.png")

**Lisons le message.** Dernière ligne :
`FileNotFoundError: [Errno 2] No such file or directory: '.../resultats/barres.png'`.

- `FileNotFoundError` : le **dossier** `resultats/` n'existe pas. `savefig` crée le fichier, **pas** le dossier.
- Le message montre le chemin complet essayé : vérifie le nom du dossier.
- Une figure vide s'affiche peut-être sous l'erreur : ignore-la.

**Correction** : enregistrer dans `sorties/` (créé à la section 0), ou créer le dossier d'abord avec
`Path("resultats").mkdir(exist_ok=True)`.

## 📌 À retenir

| Je veux… | À la main (modules 05 à 08) | Avec pandas / matplotlib |
|---|---|---|
| Charger le CSV | `csv.DictReader` + boucle + `float()` | `meteo = pd.read_csv("../donnees/meteo.csv")` |
| Regarder le tableau | `print(releves[:5])` | `meteo.head()`, `meteo.shape`, `meteo.dtypes` |
| Une colonne | boucle + `append` | `meteo["temp_max"]` ; plusieurs : `meteo[["ville", "temp_max"]]` |
| Moyenne, max, total | `sum(l) / len(l)`, `max(l)`, `sum(l)` | `.mean()`, `.max()`, `.min()`, `.sum()`, `.count()` |
| Une nouvelle colonne | boucle + calcul | `meteo["ecart"] = meteo["temp_max"] - meteo["temp_min"]` |
| Garder certaines lignes | boucle + `if` | `meteo[meteo["ville"] == "Toulouse"]` ; et : `&`, ou : `\|`, avec parenthèses |
| Trier | `sorted(liste)` | `meteo.sort_values("temp_max", ascending=False)` |
| Par ville | dict d'accumulateurs | `meteo.groupby("ville")["temp_max"].mean()` |
| Nettoyer le texte | `.strip()`, `.replace()` | `.str.strip()`, `.replace({...})`, `.str.replace(",", ".")` |
| Texte → nombre | `float(texte)` | `pd.to_numeric(colonne)` |
| Cases vides, doublons | des `if`, un `set` | `.isna()`, `.dropna()`, `.drop_duplicates()` |
| Un graphique | — | `plt.plot(x, y)`, `plt.bar(noms, valeurs)`, `plt.title/xlabel/ylabel` |
| L'enregistrer | — | `plt.savefig("sorties/nom.png")` **avant** `plt.show()` |

Lire une erreur pandas : **va à la dernière ligne**. `KeyError` = nom de colonne mal écrit ; `ValueError … ambiguous`
= `and`/`or` au lieu de `&`/`|` ; `ValueError: Unable to parse string` = texte pas convertible en nombre.

## 🧠 Mini-quiz

1. Que renvoie `meteo.shape` pour un tableau de 180 lignes et 6 colonnes ? Avec ou sans parenthèses ?
2. Pourquoi des doubles crochets dans `meteo[["ville", "temp_max"]]` ?
3. Écris le filtre qui garde les relevés d'Albi où il a plu (`pluie_mm > 0`).
4. Que fait `meteo.groupby("ville")["pluie_mm"].sum()` ? Combien de valeurs contient le résultat ?
5. Pourquoi `savefig` doit-il venir avant `show` ?

<details><summary>Réponses</summary>

1. Le tuple `(180, 6)`. **Sans** parenthèses : `shape` est un attribut, pas une méthode.
2. Les crochets extérieurs veulent dire « choisis des colonnes » ; les crochets intérieurs sont une **liste** de noms.
3. `meteo[(meteo["ville"] == "Albi") & (meteo["pluie_mm"] > 0)]` : `&` et des parenthèses autour de chaque condition.
4. La pluie totale de chaque ville : une Series de 6 valeurs, indexée par le nom des villes.
5. Dans un fichier `.py`, `show` affiche puis efface la figure : un `savefig` placé après enregistrerait une image vide.

</details>

## ➡️ Et maintenant

Ouvre `02-exercices.ipynb` : tu vas répéter chacun de ces gestes, encore et encore, jusqu'à ce qu'ils deviennent
des réflexes.